# Weather API project: Medallion architecture (Silver layer) using OpenMeteo API
### by Matias Bertuzzi
### Data Engineer | SunnyData

#1. Configuration

In [0]:
import json
from datetime import datetime, date
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType, IntegerType
)
from pyspark.testing.utils import assertDataFrameEqual
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
catalog = f"mbertuzzi_{environment}"
dbutils.widgets.text("schema", "weatherapi", "Schema")
dbutils.widgets.text("bronze_table", "bronze_weather_raw", "Bronze table name")
dbutils.widgets.text("silver_table", "silver_weather_hourly", "Silver table name")
dbutils.widgets.text("env", "dev", "Environment (dev/prod)")

In [0]:
schema = dbutils.widgets.get("schema")
bronze_table = dbutils.widgets.get("bronze_table")
silver_table = dbutils.widgets.get("silver_table")
env = dbutils.widgets.get("env")

In [0]:
bronze_table_fqn = f"{catalog}.{schema}.{bronze_table}"
silver_table_fqn = f"{catalog}.{schema}.{silver_table}"
quarantine_table_fqn = f"{catalog}.{schema}.{silver_table}_quarantine"

#2. Transformation

In [0]:
def parse_hourly_forecast(bronze_df):
    """Explode the raw Open-Meteo 'hourly' JSON array into one row per forecast hour."""
    hourly_schema = StructType([
        StructField("time", ArrayType(StringType())),
        StructField("temperature_2m", ArrayType(DoubleType())),
        StructField("relative_humidity_2m", ArrayType(DoubleType())),
        StructField("precipitation", ArrayType(DoubleType())),
        StructField("weathercode", ArrayType(IntegerType())),
    ])
    payload_schema = StructType([StructField("hourly", hourly_schema)])

    return (
        bronze_df
        .withColumn("payload", F.from_json(F.col("raw_response"), payload_schema))
        .withColumn(
            "hourly_zipped",
            F.arrays_zip(
                F.col("payload.hourly.time").alias("forecast_time"),
                F.col("payload.hourly.temperature_2m").alias("temperature_2m"),
                F.col("payload.hourly.relative_humidity_2m").alias("relative_humidity_2m"),
                F.col("payload.hourly.precipitation").alias("precipitation_mm"),
                F.col("payload.hourly.weathercode").alias("weathercode"),
            ),
        )
        .withColumn("hourly_row", F.explode("hourly_zipped"))
        .select(
            F.col("location_name"),
            F.col("latitude"),
            F.col("longitude"),
            F.to_timestamp(F.col("hourly_row.forecast_time")).alias("forecast_time"),
            F.col("hourly_row.temperature_2m").alias("temperature_2m"),
            F.col("hourly_row.relative_humidity_2m").alias("relative_humidity_2m"),
            F.col("hourly_row.precipitation_mm").alias("precipitation_mm"),
            F.col("hourly_row.weathercode").cast("int").alias("weathercode"),
            F.col("ingestion_timestamp").alias("_bronze_ingestion_timestamp"),
            F.col("source_url").alias("_source_url"),
        )
        .withColumn("load_date", F.to_date("forecast_time"))
    )

In [0]:
sample_payload = json.dumps({
    "hourly": {
        "time": ["2026-01-01T00:00", "2026-01-01T01:00"],
        "temperature_2m": [18.5, 18.1],
        "relative_humidity_2m": [70.0, 72.0],
        "precipitation": [0.0, 0.2],
        "weathercode": [1, 2],
    }
})
fixture_ingested_at = datetime(2026, 1, 1, 0, 5)

fixture_df = spark.createDataFrame(
    [("Buenos Aires", -34.6037, -58.3816, sample_payload, fixture_ingested_at, "https://fixture.test")],
    schema="location_name string, latitude double, longitude double, raw_response string, "
           "ingestion_timestamp timestamp, source_url string",
)

expected_df = spark.createDataFrame(
    [
        ("Buenos Aires", -34.6037, -58.3816, datetime(2026, 1, 1, 0, 0), 18.5, 70.0, 0.0, 1,
         fixture_ingested_at, "https://fixture.test", date(2026, 1, 1)),
        ("Buenos Aires", -34.6037, -58.3816, datetime(2026, 1, 1, 1, 0), 18.1, 72.0, 0.2, 2,
         fixture_ingested_at, "https://fixture.test", date(2026, 1, 1)),
    ],
    schema="location_name string, latitude double, longitude double, forecast_time timestamp, "
           "temperature_2m double, relative_humidity_2m double, precipitation_mm double, "
           "weathercode int, _bronze_ingestion_timestamp timestamp, _source_url string, load_date date",
)

assertDataFrameEqual(parse_hourly_forecast(fixture_df), expected_df, ignoreColumnOrder=True)
print("parse_hourly_forecast() regression test passed")

In [0]:
silver_table_exists = spark.catalog.tableExists(silver_table_fqn)

watermark = None
if silver_table_exists:
    watermark = spark.sql(
        f"SELECT MAX(_bronze_ingestion_timestamp) AS wm FROM {silver_table_fqn}"
    ).collect()[0]["wm"]

bronze_df = spark.table(bronze_table_fqn)
if watermark is not None:
    bronze_df = bronze_df.filter(F.col("ingestion_timestamp") > F.lit(watermark))

new_rows_count = bronze_df.count()
print(f"Processing {new_rows_count} new bronze row(s) since watermark={watermark}")

if new_rows_count == 0:
    dbutils.notebook.exit("No new bronze data to process — exiting silver run.")

silver_df = parse_hourly_forecast(bronze_df)

In [0]:
expectations = {
    "forecast_time_not_null": F.col("forecast_time").isNotNull(),
    "temperature_in_range": F.col("temperature_2m").between(-90, 60),
    "humidity_in_range": F.col("relative_humidity_2m").between(0, 100),
    "precipitation_non_negative": F.col("precipitation_mm") >= 0,
    "location_not_null": F.col("location_name").isNotNull(),
}

failing_condition = None
for name, cond in expectations.items():
    violation = ~cond
    violation_count = silver_df.filter(violation).count()
    print(f"Expectation '{name}': {violation_count} violating row(s)")
    failing_condition = violation if failing_condition is None else (failing_condition | violation)

valid_df = silver_df.filter(~failing_condition)
quarantine_df = silver_df.filter(failing_condition)

quarantine_count = quarantine_df.count()
if quarantine_count > 0:
    print(f"Quarantining {quarantine_count} row(s) failing one or more expectations")
    (
        quarantine_df
        .withColumn("_quarantined_at", F.current_timestamp())
        .write.format("delta").mode("append")
        .saveAsTable(quarantine_table_fqn)
    )

In [0]:
valid_df = valid_df.coalesce(1)

#3. Write

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table_fqn} (
        location_name STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        forecast_time TIMESTAMP,
        temperature_2m DOUBLE,
        relative_humidity_2m DOUBLE,
        precipitation_mm DOUBLE,
        weathercode INT,
        _bronze_ingestion_timestamp TIMESTAMP,
        _source_url STRING,
        load_date DATE
    )
    USING DELTA
    CLUSTER BY (load_date, location_name)
""")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.partitionBy(
    "location_name",
    "forecast_time"
).orderBy(
    F.col("_bronze_ingestion_timestamp").desc()
)

valid_df_dedup = (
    valid_df
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
target = DeltaTable.forName(spark, silver_table_fqn)

(
    target.alias("t")
    .merge(
        valid_df.alias("s"),
        "t.location_name = s.location_name AND t.forecast_time = s.forecast_time",
    )
    .whenMatchedUpdate(
        condition="s._bronze_ingestion_timestamp > t._bronze_ingestion_timestamp",
        set={
            "temperature_2m": "s.temperature_2m",
            "relative_humidity_2m": "s.relative_humidity_2m",
            "precipitation_mm": "s.precipitation_mm",
            "weathercode": "s.weathercode",
            "_bronze_ingestion_timestamp": "s._bronze_ingestion_timestamp",
            "_source_url": "s._source_url",
        },
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.sql(f"OPTIMIZE {silver_table_fqn}")

In [0]:
if env == "dev":
    spark.sql(f"""
        ALTER TABLE {silver_table_fqn}
        SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 24 hours')
    """)
else:
    # Prod keeps Delta's default 168h (7-day) time-travel safety window —
    # never shortened, so this property is intentionally left unset in prod.
    pass

spark.sql(f"VACUUM {silver_table_fqn}")